# Python + SQL 🐍🗄️

Python oferece um ecossistema robusto para trabalhar com bancos de dados. Nesta aula, daremos os primeiros passos com duas bibliotecas:

* [**sqlite3**](https://docs.python.org/3/library/sqlite3.html): interface para bancos SQLite, nativa do Python. É uma alternativa leve e autocontida: os dados ficam em um único arquivo e não é preciso configurar nenhum servidor. **Usamos a linguagem SQL** para conversar com o banco.
* [**SQLAlchemy** 🧙](https://www.sqlalchemy.org/): um toolkit completo para trabalhar com bancos de dados. Aqui veremos sua parte **Core**, que também executa comandos SQL, mas com uma interface mais moderna e que funciona com vários bancos diferentes.

### Base de dados exemplo

Vamos trabalhar com um banco de uma pizzaria, com apenas **duas tabelas** ligadas por **uma chave estrangeira**: cada produto pertence a uma categoria.

```
categoria
+--------------+------+
| id_categoria | PK   |
| nome         |      |
+--------------+------+

produto
+--------------+-------------------+
| id_produto   | PK                |
| desc_item    |                   |
| vl_preco     |                   |
| id_categoria | FK -> categoria   |
+--------------+-------------------+
```

> Os dados foram inspirados no dataset do Kaggle [Pizza Query](https://www.kaggle.com/datasets/teocalvo/pizzaquery/data?select=produto.csv), do curso SQL Essentials da [Linux Tips](https://www.linuxtips.io/) em parceria com [Téo Me Why](https://github.com/teomewhy).

## SQLite

### "Conectando" ao database

Como o SQLite não é um banco cliente-servidor, "conectar" significa simplesmente acessar (ou criar) o arquivo que guarda as informações do banco:

```python
import sqlite3
connection = sqlite3.connect(path)
```

O parâmetro `path` em geral é o nome do arquivo onde os registros estão (ou ficarão) armazenados. Também pode ser a string `:memory:`, que cria um banco provisório em memória, o qual deixa de existir quando a conexão é fechada.

A função a seguir abre a conexão já tratando possíveis erros. Para isso usamos a classe [`sqlite3.Error`](https://docs.python.org/3/library/sqlite3.html#sqlite3.Error), que é a base de todas as exceções dessa biblioteca.

In [ ]:
import sqlite3
from sqlite3 import Error

connection = None
try:
    connection = sqlite3.connect("pizza_app.sqlite")
    print("Conectado com sucesso!")
except Error as e:
    print(f"O erro '{e}' aconteceu")

Conectado com sucesso!


### Executando comandos SQL

O objeto `connection` possui o método `cursor()`, que cria um objeto `Cursor`. O cursor é o intermediário que envia comandos SQL ao banco e recupera os resultados (veja a [documentação](https://docs.python.org/3/library/sqlite3.html#sqlite3.Cursor)).

Seu principal método é o `execute`, que executa **uma** instrução SQL recebida como string:

```python
cursor.execute(sql)
```

#### Criando as tabelas

Primeiro criamos a tabela `categoria` e depois `produto`, que faz referência a ela através da chave estrangeira (`FOREIGN KEY`).

> Cada comando começa com `DROP TABLE IF EXISTS`, que apaga a tabela caso ela já exista. Assim você pode rodar a célula várias vezes sem erro. Repare que `produto` é apagada primeiro, pois depende de `categoria`.

In [ ]:
cursor = connection.cursor()

# Apaga as tabelas (se existirem) para podermos rodar a célula de novo
cursor.execute("DROP TABLE IF EXISTS produto")
cursor.execute("DROP TABLE IF EXISTS categoria")

# Cria a tabela categoria
cursor.execute("""
CREATE TABLE categoria (
    id_categoria INTEGER PRIMARY KEY AUTOINCREMENT,
    nome VARCHAR(50) NOT NULL
);
""")

# Cria a tabela produto
cursor.execute("""
CREATE TABLE produto (
    id_produto INTEGER PRIMARY KEY AUTOINCREMENT,
    desc_item VARCHAR(100) NOT NULL,
    vl_preco DECIMAL(10, 2),
    id_categoria INTEGER NOT NULL,
    FOREIGN KEY (id_categoria) REFERENCES categoria(id_categoria)
);
""")

#### Inserindo registros

Como as chaves primárias são `AUTOINCREMENT`, não informamos o `id`: o SQLite numera os registros automaticamente a partir do 1. Assim, `massa` é a categoria 1, `borda` é a 2, e assim por diante.

> ❗ Um `INSERT` abre uma [transação](https://docs.python.org/3/library/sqlite3.html#sqlite3-controlling-transactions) que fica pendente até chamarmos `connection.commit()`. Só então as alterações são salvas no banco de dados.

Uma **transação** é um conjunto de operações no banco que é tratado como uma unidade só: ou todas as alterações são salvas juntas (commit), ou nenhuma é (rollback). Enquanto o commit não é chamado, as alterações ficam pendentes: só a sua conexão as enxerga, e elas ainda não foram gravadas de fato no arquivo do banco.

Isso evita que o banco fique inconsistente se algo der errado no meio do caminho. Por exemplo, em uma transferência bancária o débito de uma conta e o crédito na outra precisam acontecer juntos, senão o dinheiro "some". 💰💰💰


In [ ]:
# Inserindo categorias
cursor.execute("""
INSERT INTO categoria (nome)
VALUES
('massa'),
('borda'),
('queijo'),
('ingrediente'),
('bebida');
""")

# Inserindo produtos (o último valor de cada linha é o id_categoria)
cursor.execute("""
INSERT INTO produto (desc_item, vl_preco, id_categoria)
VALUES
('tradicional', 9.25, 1),
('integral', 10.5, 1),
('tradicional', 0, 2),
('muçarela', 4, 3),
('camarão', 6, 4),
('calabresa', 5.5, 4),
('refrigerante', 5, 5);
""")

# Salva as alterações no banco
connection.commit()

### Executando queries de leitura

Depois de executar uma consulta (`SELECT`) com o `execute`, recuperamos o resultado chamando o método `fetchall()` do cursor. Ele retorna uma **lista de tuplas**, em que cada tupla é um registro do banco:

```python
fetchall() -> list[tuple]
```

> Existem outros métodos de leitura, como `fetchone` e `fetchmany`. Veja como eles funcionam na documentação: [`fetchone`](https://docs.python.org/3/library/sqlite3.html#sqlite3.Cursor.fetchone) e [`fetchmany`](https://docs.python.org/3/library/sqlite3.html#sqlite3.Cursor.fetchmany).

Nossa primeira consulta vai verificar as tabelas que criamos. Todo banco SQLite tem uma "tabela de esquema", chamada `sqlite_master`, que guarda a estrutura do banco (veja a [documentação](https://www.sqlite.org/schematab.html)). Vamos ler o nome de todas as tabelas registradas nela:

In [ ]:
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()
print(tables)

[('categoria',), ('sqlite_sequence',), ('produto',)]


Além das nossas tabelas, aparece uma extra: a [`sqlite_sequence`](https://www.sqlite.org/fileformat2.html#seqtab). Ela é uma tabela interna, criada automaticamente pelo SQLite para controlar o `AUTOINCREMENT`.

Agora vamos ler o conteúdo das nossas tabelas:

In [ ]:
cursor.execute("SELECT * FROM sqlite_sequence")
print(cursor.fetchall())

[('categoria', 5), ('produto', 7)]


In [ ]:
cursor.execute("SELECT * FROM categoria")
print(cursor.fetchall())

[(1, 'massa'), (2, 'borda'), (3, 'queijo'), (4, 'ingrediente'), (5, 'bebida')]


In [ ]:
cursor.execute("SELECT * FROM produto")
produtos = cursor.fetchall()

for produto in produtos:
    print(produto)

(1, 'tradicional', 9.25, 1)
(2, 'integral', 10.5, 1)
(3, 'tradicional', 0, 2)
(4, 'muçarela', 4, 3)
(5, 'camarão', 6, 4)
(6, 'calabresa', 5.5, 4)
(7, 'refrigerante', 5, 5)


#### Usando a chave estrangeira: `JOIN`

A chave estrangeira permite combinar as duas tabelas. O `JOIN` a seguir junta cada produto ao nome da sua categoria:

In [ ]:
cursor.execute("""
SELECT produto.desc_item, categoria.nome, produto.vl_preco
FROM produto
JOIN categoria ON produto.id_categoria = categoria.id_categoria;
""")

for linha in cursor.fetchall():
    print(linha)

('tradicional', 'massa', 9.25)
('integral', 'massa', 10.5)
('tradicional', 'borda', 0)
('muçarela', 'queijo', 4)
('camarão', 'ingrediente', 6)
('calabresa', 'ingrediente', 5.5)
('refrigerante', 'bebida', 5)


Terminamos com o SQLite. Sempre que acabar de usar o banco, feche a conexão:

In [ ]:
connection.close()

### ✏️ Exercício: SQLite

Usando o `sqlite3`, crie, preencha e consulte uma nova tabela:

1. Crie uma nova conexão com um banco de dados chamado `clientes.db` e já salva o cursor da conexão em uma variável
2. Crie a tabela `cliente` com as colunas `id_cliente` (chave primária, `INTEGER PRIMARY KEY AUTOINCREMENT`), `nome` (`TEXT`) e `cidade` (`TEXT`).
3. Insira 3 clientes. Não esqueça do `commit`!
4. Consulte todos os registros da tabela e imprima cada um.

> Dica: comece com `DROP TABLE IF EXISTS cliente` para poder rodar a célula mais de uma vez.

In [ ]:
import sqlite3

coneccao = sqlite3.connect("clientes.db")
cursor = coneccao.cursor()

cursor.execute("DROP TABLE IF EXISTS cliente")

cursor.execute("""CREATE TABLE cliente(
id_cliente INTEGER PRIMARY KEY AUTOINCREMENT,
nome VARCHAR(100) NOT NULL,
cidade VARCHAR(100) NOT NULL
);
""")

cursor.execute("""
INSERT INTO cliente (nome, cidade)
VALUES
('George', 'Ribeirão das Neves'),
('Tamiris', 'Rio de Janeiro'),
('Pedro', 'Belo Horizonte');
""")

coneccao.commit()

cursor.execute("SELECT * FROM cliente")
print(cursor.fetchall())

coneccao.close()




[(1, 'George', 'Ribeirão das Neves'), (2, 'Tamiris', 'Rio de Janeiro'), (3, 'Pedro', 'Belo Horizonte')]


---
# SQLAlchemy 🧙

No [principal tutorial do SQLAlchemy](https://docs.sqlalchemy.org/en/20/tutorial/index.html) são apresentados os dois principais elementos do toolkit:
* **SQLAlchemy Core**: ferramentas para conexão e interação com bancos de dados através de instruções SQL.
* **SQLAlchemy ORM**: mapeamento objeto-relacional. É uma extensão do Core, que abstrai os esquemas e as interações com o banco usando orientação a objetos.

Nesta aula vamos ver o **Core**, usando novamente o SQLite.

## SQLAlchemy Core

### Estabelecendo uma conexão
Para estabelecer uma conexão com um banco de dados usando SQLAlchemy, alimentamos o método `create_engine` com uma URL que segue uma estrutura específica. [Na documentação](https://docs.sqlalchemy.org/en/20/core/engines.html#database-urls) você vê em detalhes a anatomia dessa URL, também apresentada a seguir:
```python
create_engine("<dialect>+<driver>://<hostname>/<database>")
```

* `dialect`: dialeto de comunicação. Essencialmente, qual banco será utilizado (sqlite, mysql, postgres, etc.).
* `driver`: API de comunicação desejada.
* `hostname`: endereço de conexão (veja a seguir sua estrutura).
* `database`: nome ou caminho para o banco de dados.

**SQLite** <br>
Usando o driver nativo do SQLite no Python (módulo `sqlite3`), a URL de criação da conexão é:
```python
# engine = create_engine("sqlite+pysqlite://<nohostname>/<database>")
engine = create_engine("sqlite+pysqlite:///<nome_do_arquivo>")
```

> O parâmetro `echo=True` faz o SQLAlchemy imprimir os comandos SQL que ele executa, o que ajuda a entender o que acontece por trás.

In [ ]:
from sqlalchemy import create_engine
from sqlalchemy import text

engine = create_engine("sqlite+pysqlite:///exemplo.db", echo=True)

### Transações

Existem duas abordagens para gerenciar [transações](https://docs.sqlalchemy.org/en/20/tutorial/dbapi_transactions.html#working-with-transactions-and-the-dbapi).

Na abordagem "*commit as you go*", usamos o método `engine.connect()`, que retorna o objeto da conexão `conn`, e precisamos chamar explicitamente `conn.commit()` ao final de cada transação.

In [ ]:
with engine.connect() as conn:
    conn.execute(text("CREATE TABLE ponto (x int, y int)"))
    conn.execute(
        text("INSERT INTO ponto (x, y) VALUES (:x, :y)"),
        [{"x": 1, "y": 1}, {"x": 2, "y": 4}],
    )
    conn.commit()

2026-09-30 17:59:46,334 INFO sqlalchemy.engine.Engine BEGIN (implicit)


INFO:sqlalchemy.engine.Engine:BEGIN (implicit)


2026-09-30 17:59:46,339 INFO sqlalchemy.engine.Engine CREATE TABLE ponto (x int, y int)


INFO:sqlalchemy.engine.Engine:CREATE TABLE ponto (x int, y int)


2026-09-30 17:59:46,342 INFO sqlalchemy.engine.Engine [generated in 0.00813s] ()


...

INFO:sqlalchemy.engine.Engine:INSERT INTO ponto (x, y) VALUES (?, ?)


2026-09-30 17:59:46,367 INFO sqlalchemy.engine.Engine [generated in 0.00218s] [(1, 1), (2, 4)]


INFO:sqlalchemy.engine.Engine:[generated in 0.00218s] [(1, 1), (2, 4)]


2026-09-30 17:59:46,371 INFO sqlalchemy.engine.Engine COMMIT


INFO:sqlalchemy.engine.Engine:COMMIT


Na abordagem "*begin once*", utilizamos o método `engine.begin()` para criar um contexto (`with`) que, se a transação for bem-sucedida, executa `commit()` automaticamente; caso contrário, executa `rollback` se uma exceção for lançada.

In [ ]:
with engine.begin() as conn:
    conn.execute(
        text("INSERT INTO ponto (x, y) VALUES (:x, :y)"),
        [{"x": 6, "y": 8}, {"x": 9, "y": 10}],
    )
    # o commit é feito implicitamente ao final do contexto with

2026-09-30 17:59:51,525 INFO sqlalchemy.engine.Engine BEGIN (implicit)


INFO:sqlalchemy.engine.Engine:BEGIN (implicit)


2026-09-30 17:59:51,531 INFO sqlalchemy.engine.Engine INSERT INTO ponto (x, y) VALUES (?, ?)


INFO:sqlalchemy.engine.Engine:INSERT INTO ponto (x, y) VALUES (?, ?)


2026-09-30 17:59:51,533 INFO sqlalchemy.engine.Engine [cached since 5.168s ago] [(6, 8), (9, 10)]


INFO:sqlalchemy.engine.Engine:[cached since 5.168s ago] [(6, 8), (9, 10)]


2026-09-30 17:59:51,538 INFO sqlalchemy.engine.Engine COMMIT


INFO:sqlalchemy.engine.Engine:COMMIT


### Comandos de leitura

Chamadas a `conn.execute()` retornam um objeto `Cursor`, semelhante ao que vimos com o módulo `sqlite3`. Podemos invocar métodos de leitura, como `fetchall`, para recuperar os elementos da consulta. No contexto do SQLAlchemy, cada registro é um objeto do tipo `Row`, cujos elementos podem ser acessados por nome ou por índice.

In [ ]:
with engine.connect() as conn:
    cursor = conn.execute(
        text("SELECT x, y FROM ponto WHERE y > :y"),
        {"y": 2}
    )

    result = cursor.fetchall()
    print(result, type(result[0]), '\n')

    print('Acesso nomeado e indexado')
    for row in result:
        print(f"row.x: {row.x}\t row.y: {row.y}")
        print(f"row[0]: {row[0]}\t row[1]: {row[1]}\n")


2026-09-30 18:00:53,506 INFO sqlalchemy.engine.Engine BEGIN (implicit)


INFO:sqlalchemy.engine.Engine:BEGIN (implicit)


2026-09-30 18:00:53,510 INFO sqlalchemy.engine.Engine SELECT x, y FROM ponto WHERE y > ?


INFO:sqlalchemy.engine.Engine:SELECT x, y FROM ponto WHERE y > ?


2026-09-30 18:00:53,512 INFO sqlalchemy.engine.Engine [generated in 0.00574s] (2,)


INFO:sqlalchemy.engine.Engine:[generated in 0.00574s] (2,)


[(2, 4), (6, 8), (9, 10)] <class 'sqlalchemy.engine.row.Row'> 

Acesso nomeado e indexado
row.x: 2	 row.y: 4
row[0]: 2	 row[1]: 4

row.x: 6	 row.y: 8
row[0]: 6	 row[1]: 8

row.x: 9	 row.y: 10
row[0]: 9	 row[1]: 10

2026-09-30 18:00:53,516 INFO sqlalchemy.engine.Engine ROLLBACK


INFO:sqlalchemy.engine.Engine:ROLLBACK


### Dropando a tabela exemplo...

In [ ]:
with engine.begin() as conn:
    cursor = conn.execute(text("DROP TABLE ponto"))

2026-09-30 18:01:05,436 INFO sqlalchemy.engine.Engine BEGIN (implicit)


INFO:sqlalchemy.engine.Engine:BEGIN (implicit)


2026-09-30 18:01:05,440 INFO sqlalchemy.engine.Engine DROP TABLE ponto


INFO:sqlalchemy.engine.Engine:DROP TABLE ponto


2026-09-30 18:01:05,442 INFO sqlalchemy.engine.Engine [generated in 0.00243s] ()


INFO:sqlalchemy.engine.Engine:[generated in 0.00243s] ()


2026-09-30 18:01:05,459 INFO sqlalchemy.engine.Engine COMMIT


INFO:sqlalchemy.engine.Engine:COMMIT


Fechando a conexão...

In [ ]:
conn.close()

### ✏️ Exercício: SQLAlchemy

Refaça o mesmo exercício do SQLite, agora usando o SQLAlchemy (com o `engine` já criado):

1. Crie uma nova conexão com um banco de dados chamado `clientes_alchemy.db` e já salva o cursor da conexão em uma variável
2. Crie a tabela `cliente` com as colunas `id_cliente` (chave primária, `INTEGER PRIMARY KEY AUTOINCREMENT`), `nome` (`TEXT`) e `cidade` (`TEXT`).
3. Insira 3 clientes.
4. Consulte todos os registros da tabela e imprima cada um.

> Dicas: use `engine.begin()` para criar e inserir (assim o commit é automático) e `engine.connect()` para a consulta. Comece com `DROP TABLE IF EXISTS cliente`, pois a tabela pode já existir do exercício anterior.

In [5]:
from sqlalchemy import create_engine
from sqlalchemy import text

engine = create_engine("sqlite+pysqlite:///clientes_alchemy.db")

with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS clientes"))
    conn.execute(text("CREATE TABLE clientes(id_cliente INTEGER PRIMARY KEY AUTOINCREMENT, nome TEXT, cidade TEXT)"))
    conn.execute(text("INSERT INTO clientes (nome, cidade) VALUES ('Pedro', 'Ribeirão das Neves')"))
    conn.execute(text("INSERT INTO clientes (nome, cidade) VALUES ('Tamiris', 'Rio de Janeiro')"))
    conn.execute(text("INSERT INTO clientes (nome, cidade) VALUES ('George', 'Belo Horizonte')"))

    cursor = conn.execute(text("SELECT * FROM clientes"))

    for linha in cursor.fetchall():
        print(linha)

(1, 'Pedro', 'Ribeirão das Neves')
(2, 'Tamiris', 'Rio de Janeiro')
(3, 'George', 'Belo Horizonte')
